# Weather Station History Analysis
This notebook queries the local Parquet data lake to find the earliest recorded data and the total history length for each weather station.

In [1]:
import pyarrow.dataset as ds
import pandas as pd

# Load the partitioned Parquet dataset
dataset = ds.dataset('../data/dwd_historical_lake', format='parquet', partitioning='hive')
df = dataset.to_table(columns=['Station_ID', 'Date', 'Bundesland']).to_pandas()

# Aggregate by station to find earliest date and history length
summary = df.groupby(['Station_ID', 'Bundesland']).agg(
    Earliest_Date=('Date', 'min'),
    Latest_Date=('Date', 'max'),
    Days_of_History=('Date', 'count')
).reset_index()

# Calculate years of history and sort by earliest date
summary['Years_of_History'] = (summary['Days_of_History'] / 365.25).round(1)
summary = summary.sort_values('Earliest_Date')

# Display the top 50 oldest stations
summary.head(50)

,Station_ID,Bundesland,Earliest_Date,Latest_Date,Days_of_History,Years_of_History
561,02928,Sachsen,1759-11-01,2025-12-31,72775,199.2
435,02290,Bayern,1781-01-01,2025-12-31,87642,240.0
1090,05906,Baden-Wuerttemberg,1781-01-01,2025-12-31,34334,94.0
465,02444,Thueringen,1821-01-01,2025-12-31,73272,200.6
296,01425,Hessen,1826-01-01,1961-11-30,49550,135.7
210,01047,Sachsen,1828-01-01,1915-12-31,21549,59.0
373,01960,Sachsen-Anhalt,1851-01-01,1975-12-31,45655,125.0
767,04024,Mecklenburg-Vorpommern,1853-07-01,2025-12-31,59606,163.2
342,01691,Niedersachsen,1858-01-01,2025-12-31,59690,163.4
701,03631,Niedersachsen,1858-03-01,2025-12-31,50728,138.9


In [2]:
# Filter for stations with at least 100 years of history
stations_over_100 = summary[summary['Years_of_History'] >= 100]
print(f"Number of stations with >= 100 years of history: {len(stations_over_100)}")

Number of stations with >= 100 years of history: 47
